# Laboratorios de pandas: preparar y estructurar los datos de Instacart

**Aprendiz:** Juan Camilo Montes Velásquez
**Programa:** Tecnólogo en Análisis y Desarrollo de Software, SENA
**Evidencia:** GA1-220501094-01-AA1-EV06 – Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este cuaderno desarrolla, en orden, las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python*: los ejemplos de cada explicación, el laboratorio guiado y los 7 ejercicios del reto. Cada lección usa las variables de la anterior, así que el cuaderno debe ejecutarse de principio a fin (*Run → Run All Cells*).

Los CSV originales están en `data/raw/` y los archivos que genera este cuaderno se guardan en `data/processed/`.

## Preparación: entorno y carga de datos

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(exist_ok=True)

print('pandas', pd.__version__)

pandas 3.1.1


La carga es la misma de la Lección 1 (actividad anterior): los archivos usan punto y coma como separador. Las lecciones siguientes trabajan sobre estas variables.

In [2]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


# Lección 2: Limpieza y estandarización de datos crudos

Orden de trabajo: duplicados, tipos, nulos, texto, duplicados implícitos y verificación. Para cada problema se decide si eliminar, corregir o conservar, según lo que significa el dato.

### Duplicados exactos: detectar con `duplicated` y eliminar con `drop_duplicates`

In [3]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


Hay **30 filas**: 15 pedidos y sus 15 copias. Todas tienen `order_dow = 3` y `order_hour_of_day = 2` (miércoles a las 2 a. m.). Que ocurran todas en la misma franja apunta a un fallo del sistema, no a pedidos reales, así que es seguro borrarlas.

In [4]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


### Tipos de dato con `astype`

In [5]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB


Después: 7.2 MB


Con tipos más pequeños `orders` pasa de **23,0 MB a 7,2 MB** sin perder información. `Int16` con mayúscula es un entero *nullable*: admite enteros y nulos a la vez, por eso sirve para `days_since_prior_order` y `add_to_cart_order`.

### Nulos: comprobar la causa antes de decidir

In [6]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


In [7]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


| Columna | Causa | Decisión |
|---|---|---|
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo (rellenar con 0 inventaría una compra el mismo día) |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el reto | Conservar el nulo |

### Texto con los métodos `.str`

In [8]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16


0 0


### Duplicados implícitos

In [9]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141


268 de 280


Hay **141 nombres repetidos** después de estandarizar, y 268 de los 280 productos implicados sí aparecen en pedidos. Decisión: **conservar y documentar**. Tienen `product_id` distintos y borrarlos dejaría pedidos apuntando a productos inexistentes; en los análisis se agrupa por `product_id`, no por nombre.

### Limpieza reutilizable con funciones

In [10]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


Verificación de toda la limpieza (si la celda no muestra nada, la limpieza está completa):

In [11]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'

## Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Un pipeline es una secuencia de pasos de limpieza encadenados que parte del archivo crudo, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso.

**Paso 1: recargar los datos crudos**

In [12]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


**Paso 2: una función que anota el estado de la tabla**

In [13]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

**Paso 3: un paso de limpieza por función**

In [14]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

**Paso 4: encadenar los pasos con `pipe`**

In [15]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

**Paso 5: el informe de cambios** (se guarda también en `data/processed/cleaning_report.csv`)

In [16]:
informe = pd.DataFrame(registro)
informe.to_csv(SALIDA / 'cleaning_report.csv', index=False)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:** quitar duplicados eliminó 15 filas y 2 nulos (dos de los pedidos repetidos eran primeros pedidos). Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB. Los 836 nulos de `order_prods` siguen ahí a propósito, porque se decidió conservarlos.

**Paso 6: validar el resultado**

In [17]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


**Conclusiones del laboratorio 2**
- El pipeline es una cadena de funciones pequeñas que reciben y devuelven un DataFrame, unidas con `pipe`.
- La función `anotar` produce el informe de cambios sin esfuerzo extra y responde con cifras a "¿qué le hiciste a los datos?".
- `validar` convierte las reglas de calidad en una prueba que se ejecuta cada vez; el pipeline produce exactamente las mismas tablas que la limpieza manual.

## Reto 2

**1. ¿`order_prods` tiene filas duplicadas exactas? ¿Se repite algún par `order_id` + `product_id`? ¿`reordered` solo contiene 0 y 1?**

In [18]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique()))

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [np.int8(0), np.int8(1)]


**Respuesta:** `order_prods` no tiene filas duplicadas exactas ni pares `order_id` + `product_id` repetidos: cada producto aparece una sola vez por pedido. `reordered` solo contiene 0 y 1. Esta tabla no necesita limpieza de duplicados.

**2. ¿`aisles` y `departments` tienen nombres duplicados o con espacios al inicio o al final? ¿Necesitan limpieza de texto?**

In [19]:
for nombre, serie in [('aisles', aisles['aisle']), ('departments', departments['department'])]:
    print(nombre)
    print('  nombres duplicados:', serie.duplicated().sum())
    print('  con espacios al inicio o al final:', (serie != serie.str.strip()).sum())
    print('  con espacios dobles:', serie.str.contains('  ', regex=False).sum())
    print('  con mayúsculas:', (serie != serie.str.lower()).sum())

aisles
  nombres duplicados: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
  con mayúsculas: 0
departments
  nombres duplicados: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
  con mayúsculas: 0


**Respuesta:** ni `aisles` ni `departments` tienen nombres duplicados, espacios al inicio o al final, espacios dobles ni mayúsculas. **No necesitan limpieza de texto.** Lo único a tener en cuenta es el valor centinela `missing` (pasillo 100 y departamento 21), que se trata en el punto 7.

**3. Los 836 nulos de `add_to_cart_order`: ¿en cuántos pedidos ocurren, cuántos productos tiene como mínimo cada uno y qué los explica?**

In [20]:
filas_nulas = order_prods[order_prods['add_to_cart_order'].isna()]
pedidos_con_nulos = filas_nulas['order_id'].unique()
print('Pedidos con nulos:', len(pedidos_con_nulos))

detalle_nulos = order_prods[order_prods['order_id'].isin(pedidos_con_nulos)]
tamano = detalle_nulos.groupby('order_id').size()
print('Productos por pedido -> mínimo:', tamano.min(), '| máximo:', tamano.max())
print('Posición más alta registrada en esos pedidos:', detalle_nulos['add_to_cart_order'].max())
print('Posición más alta en todo order_prods:', order_prods['add_to_cart_order'].max())

Pedidos con nulos: 70
Productos por pedido -> mínimo: 65 | máximo: 127
Posición más alta registrada en esos pedidos: 64
Posición más alta en todo order_prods: 64


**Respuesta:** los 836 nulos ocurren en **70 pedidos**, y cada uno tiene **como mínimo 65 productos** (el más grande, 127). La posición más alta registrada en todo `order_prods` es 64. Esto indica que **el sistema solo guarda la posición del carrito hasta 64**: los productos agregados después del 64 quedan sin posición. No es un error de datos sino un límite del registro, así que se conserva el nulo.

**4. Cargar de nuevo `products.csv` en `products_crudo` y contar cuántos nombres (no nulos) cambiaron con la estandarización.**

In [21]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')

con_nombre = products_crudo['product_name'].notna()
cambiaron = products_crudo.loc[con_nombre, 'product_name'] != products.loc[con_nombre, 'product_name']
print('Nombres con valor original:', con_nombre.sum())
print('Nombres que cambiaron:', cambiaron.sum())
print('Solo por espacios (mismo texto en minúsculas):',
      (products_crudo.loc[con_nombre, 'product_name'].str.replace('\xa0', ' ', regex=False)
       .str.replace(r'\s+', ' ', regex=True).str.strip().str.lower()
       == products.loc[con_nombre, 'product_name'].str.lower())[cambiaron].sum())

Nombres con valor original: 48436
Nombres que cambiaron: 6321


Solo por espacios (mismo texto en minúsculas): 6321


**Respuesta:** de 48 436 nombres con valor original, **6 321 cambiaron** con la estandarización. En todos los casos el cambio fue solo de espacios y de mayúsculas/minúsculas: escritos en minúsculas y con espacios normalizados, los nombres originales y los limpios son iguales. La limpieza no alteró el contenido de ningún nombre.

**5. Serie con el nombre del día de cada pedido convertida a categoría ordenada de domingo a sábado. Comparar la memoria del texto con la de la categoría.**

In [22]:
orden_dias = ['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']
dia_texto = orders['order_dow'].map(dict(enumerate(orden_dias)))
dia_categoria = pd.Series(pd.Categorical(dia_texto, categories=orden_dias, ordered=True))

mem_texto = dia_texto.memory_usage(deep=True) / 1e6
mem_cat = dia_categoria.memory_usage(deep=True) / 1e6
print(f'Texto: {mem_texto:.2f} MB | Categoría: {mem_cat:.2f} MB | {mem_texto / mem_cat:.0f} veces menos')
print(dia_categoria.cat.categories.tolist())
print(dia_categoria.min(), '->', dia_categoria.max())

Texto: 7.14 MB | Categoría: 0.48 MB | 15 veces menos
['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']
domingo -> sábado


**Respuesta:** como texto, la Serie ocupa **7,14 MB**; como categoría ordenada ocupa **0,48 MB**, unas **15 veces menos**. La categoría guarda los 7 nombres una sola vez y en cada fila solo un código numérico. Además, al ser ordenada, pandas sabe que domingo va antes que sábado (`min` = domingo, `max` = sábado), lo que sirve para ordenar y graficar en el orden correcto.

**6. ¿Cuántas filas de `order_prods` corresponden a productos `'Unknown'` y en cuántos pedidos aparecen? ¿Se eliminarían?**

In [23]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods['product_id'].isin(ids_unknown)
print('Filas:', filas_unknown.sum(), f'({filas_unknown.mean() * 100:.2f} % de order_prods)')
print('Pedidos en los que aparecen:', order_prods.loc[filas_unknown, 'order_id'].nunique())

Filas: 9547 (0.21 % de order_prods)
Pedidos en los que aparecen: 8254


**Respuesta:** hay **9 547 filas** de productos `'Unknown'` (0,21 % de `order_prods`) repartidas en **8 254 pedidos**. **No las eliminaría:** son compras reales, y borrarlas cambiaría el tamaño de 8 254 pedidos y la tasa de recompra. Basta con conservarlas y, cuando se analice por producto o departamento, tener presente que pertenecen a `missing`.

**7. En una copia de `departments`, reemplazar `'missing'` por un nulo real (`pd.NA`). ¿Por qué no conviene hacerlo en la tabla original antes de agrupar?**

In [24]:
departments_na = departments.copy()
departments_na['department'] = departments_na['department'].replace('missing', pd.NA)
print(departments_na.tail(3))
print('Nulos:', departments_na['department'].isna().sum())

prueba = products.merge(departments_na, on='department_id')
print('Productos al agrupar con "missing":', products.merge(departments, on='department_id').groupby('department').size().sum())
print('Productos al agrupar con pd.NA:     ', prueba.groupby('department').size().sum())

    department_id department
18             19     snacks
19             20       deli
20             21        NaN
Nulos: 1
Productos al agrupar con "missing": 49694
Productos al agrupar con pd.NA:      48436


**Respuesta:** en la copia, el departamento 21 queda como nulo real. No conviene hacerlo en la tabla original antes de agrupar porque **`groupby` descarta las filas cuya clave es nula**: al agrupar por departamento, los 1 258 productos de `missing` desaparecerían en silencio (49 694 frente a 48 436), y los totales dejarían de cuadrar.

# Lección 3: Filtrado, ordenamiento y agrupación de datos

Antes de escribir código, cada pregunta se traduce a "métrica por grupo". Por ejemplo, "¿a qué hora se compra más?" es "número de pedidos por hora".

### Filtrado con máscaras booleanas

In [25]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


In [26]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

45599


### Filtrado con `query`

In [27]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

1449


### Ordenamiento con `sort_values` y `nlargest`

In [28]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100


        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


### Agrupación con `groupby`

In [29]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

10 40578


In [30]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


**Interpretación:** el domingo (0) y el lunes (1) concentran la mayor cantidad de pedidos y de clientes. La espera media entre pedidos es algo más alta el fin de semana (11,8 días el domingo) que a mitad de semana (10,5 días el jueves y el viernes). La media ignora los nulos, por eso fue correcto no rellenarlos con 0.

### Tabla dinámica con `pivot_table`

In [31]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


### Valores por fila con `transform`

In [32]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')

231946 30527


## Laboratorio guiado 3: perfil de hábitos de compra

Mercadeo quiere campañas según qué tan seguido compra cada cliente y a qué hora compra cada tipo de cliente. Plan: contar pedidos por cliente, segmentar clientes, clasificar pedidos en franjas horarias y cruzar segmentos con franjas.

**Paso 1: pedidos por cliente**

In [33]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

**Paso 2: segmentar clientes con `pd.cut`** (ocasional: 1 pedido, regular: 2 a 4, frecuente: 5 o más)

In [34]:
segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


**Paso 3: llevar el segmento a cada pedido con `map`**

In [35]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


**Paso 4: resumen por segmento**

In [36]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


**Interpretación:** el 19,4 % de los clientes (los frecuentes) hace casi la mitad de los pedidos (231 946 de 478 952) y vuelve cada 6 días (mediana). Los ocasionales también tienen espera porque su único pedido en estos datos no es necesariamente el primero que hicieron en la tienda.

**Paso 5: franjas horarias y tabla cruzada**

In [37]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


**Interpretación:** todos los segmentos compran sobre todo en la tarde, pero los frecuentes compran más en la mañana (35,2 % frente a 30,0 % de los ocasionales). Para mercadeo, una campaña de fidelización debería llegar en la mañana.

**Paso 6: las combinaciones de día y franja más activas**

In [38]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

**Paso 7: dejar `orders` como estaba**

In [39]:
orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

(478952, 6)


**Conclusiones del laboratorio 3**
- Una pregunta de negocio compleja se resuelve en pasos: agrupar, categorizar, devolver al detalle y cruzar.
- `pd.cut` convierte números en categorías con límites explícitos, y `map` lleva un resultado calculado por cliente a cada pedido.
- `crosstab` con `normalize` permite comparar grupos de distinto tamaño: los clientes frecuentes compran más en la mañana, así que una campaña de fidelización debería llegar en ese horario.

## Reto 3

**1. Métrica por grupo: pedidos por hora, solo los domingos. ¿Cuáles son las tres horas con más pedidos?**

In [40]:
domingos = orders[orders['order_dow'] == 0]
domingos.groupby('order_hour_of_day').size().nlargest(3)

order_hour_of_day
15    7585
14    7533
13    7483
dtype: int64

**Respuesta** (*pedidos por hora, solo domingos*): las tres horas con más pedidos los domingos son las **15 h (7 585)**, las **14 h (7 533)** y las **13 h (7 483)**. A diferencia del total de la semana, cuya hora pico es las 10 a. m., el domingo se compra más a primera hora de la tarde.

**2. Métrica por grupo: espera media (`days_since_prior_order`) por hora. ¿A qué hora es mayor y a qué hora es menor?**

In [41]:
espera_por_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean().round(2)
print('Mayor:', espera_por_hora.idxmax(), 'h ->', espera_por_hora.max(), 'días')
print('Menor:', espera_por_hora.idxmin(), 'h ->', espera_por_hora.min(), 'días')

Mayor: 0 h -> 12.52 días
Menor: 22 h -> 10.62 días


**Respuesta** (*espera media por hora*): la espera media es **mayor a las 0 h (12,52 días)** y **menor a las 22 h (10,62 días)**. La diferencia es de casi 2 días: quienes compran a medianoche tardan más en volver.

**3. Condición: clientes con 10 pedidos o más. ¿Cuántos son?**

In [42]:
pedidos_por_cliente = orders.groupby('user_id').size()
print((pedidos_por_cliente >= 10).sum())

6327


**Respuesta** (*clientes con 10 pedidos o más*): **6 327 clientes** tienen 10 pedidos o más en este archivo, apenas el 4 % de los 157 437 clientes.

**4. Condición: pedidos hechos exactamente 7 días después del anterior. ¿Cuántos son y qué porcentaje del total representan?**

In [43]:
a_los_7 = orders['days_since_prior_order'] == 7
print(a_los_7.sum(), f'{a_los_7.sum() / len(orders) * 100:.2f} %')
print('Valor más frecuente de days_since_prior_order:', orders['days_since_prior_order'].mode()[0])

44577 9.31 %
Valor más frecuente de days_since_prior_order: 30


**Respuesta** (*pedidos con `days_since_prior_order == 7`*): **44 577 pedidos**, el **9,31 %** del total. La compra semanal es un hábito marcado. Aun así, el valor más frecuente es 30, porque ese campo está topado en 30 días (todo lo que pasa de un mes se registra como 30).

**5. Métrica por grupo con `order_prods`: productos por pedido. ¿Cuántos pedidos tienen 20 productos o más? ¿Cuál es el pedido más grande?**

In [44]:
productos_por_pedido = order_prods.groupby('order_id').size()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum())
print('Pedido más grande:', productos_por_pedido.idxmax(), 'con', productos_por_pedido.max(), 'productos')

Pedidos con 20 productos o más: 49420
Pedido más grande: 61355 con 127 productos


**Respuesta** (*productos por pedido*): **49 420 pedidos** tienen 20 productos o más. El pedido más grande es el **61355, con 127 productos**; es uno de los 70 pedidos con nulos en `add_to_cart_order` del Reto 2.

**6. Métrica por grupo con `order_prods`: tasa de recompra (media de `reordered`) por posición del carrito, de la 1 a la 5. ¿Qué patrón se ve?**

In [45]:
primeras_posiciones = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
primeras_posiciones.groupby('add_to_cart_order')['reordered'].mean().round(3)

add_to_cart_order
1    0.679
2    0.677
3    0.659
4    0.638
5    0.617
Name: reordered, dtype: float64

**Respuesta** (*tasa de recompra por posición 1 a 5*): la tasa baja a medida que avanza la posición: **0,679** en la posición 1, 0,677 en la 2, 0,659 en la 3, 0,638 en la 4 y **0,617** en la 5. Los clientes agregan primero lo que compran siempre y después los productos nuevos o de prueba.

**7. Métrica por grupo: mediana de `days_since_prior_order` en fin de semana (domingo y sábado) frente al resto de la semana.**

In [46]:
fin_de_semana = orders['order_dow'].isin([0, 6]).rename('fin_de_semana')
orders.groupby(fin_de_semana)['days_since_prior_order'].agg(['median', 'mean', 'count']).round(2)

,median,mean,count
fin_de_semana,,,
False,7.0,10.87,312767
True,8.0,11.62,137368


**Respuesta** (*mediana de espera, fin de semana vs. resto*): la mediana es de **8 días en fin de semana** y **7 días entre semana** (medias de 11,62 y 10,87). Quienes compran el sábado o el domingo tardan un poco más en volver.